In [ ]:
"""Phase 6: The Oracle Pipeline (v14) - Road to 95%+.

This architecture combines:
1. GNN Spatial Embeddings (Graph Structural Intelligence)
2. Zero-Inflated Two-Stage Model (Classification -> Regression)
3. "The Oracle" Time-Series Strategy: Uses Random Splitting to force the model 
   to learn exact daily seasonality (`geohash_hour_te`). Since the dataset only 
   contains 2 days, this proxy for 'demand yesterday' is the strongest signal 
   possible for predicting the test set.
"""

from __future__ import annotations

import argparse
import math
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-gridlock")

import catboost as cb
import geohash
import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = "demand"
ZERO_THRESHOLD = 0.015


def parse_timestamp(value: str) -> tuple[int, int]:
    hour, minute = str(value).split(":")
    return int(hour), int(minute)

def decode_geohash(value: str) -> tuple[float, float]:
    lat, lon = geohash.decode(str(value))
    return float(lat), float(lon)

def seed_everything(seed: int) -> None:
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# --- 1. PyTorch GNN for Spatial Embeddings ---
class GCN(nn.Module):
    def __init__(self, in_features: int, hidden_dim: int = 16):
        super().__init__()
        self.gc1 = nn.Linear(in_features, hidden_dim)
        self.gc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h1 = torch.relu(self.gc1(torch.matmul(adj, x)))
        emb = torch.relu(self.gc2(torch.matmul(adj, h1)))
        pred = self.out(emb)
        return emb, pred

def get_gnn_embeddings(train_df: pd.DataFrame, test_df: pd.DataFrame, train_target_df: pd.DataFrame) -> dict[str, np.ndarray]:
    print("Building GNN Spatial Embeddings...")
    all_gh = sorted(list(set(train_df["geohash"].unique()) | set(test_df["geohash"].unique())))
    gh_to_idx = {gh: i for i, gh in enumerate(all_gh)}
    N = len(all_gh)
    
    adj = np.eye(N, dtype=np.float32)
    for gh in all_gh:
        idx = gh_to_idx[gh]
        for n_gh in geohash.neighbors(gh):
            if n_gh in gh_to_idx:
                adj[idx, gh_to_idx[n_gh]] = 1.0
                
    deg = adj.sum(axis=1)
    d_inv_sqrt = np.power(np.maximum(deg, 1.0), -0.5)
    adj_norm = d_inv_sqrt[:, None] * adj * d_inv_sqrt[None, :]
    
    device = torch.device("cpu")
    adj_t = torch.from_numpy(adj_norm).float().to(device)

    X = np.zeros((N, 3), dtype=np.float32)
    gh_counts = train_target_df["geohash"].value_counts().to_dict()
    for gh in all_gh:
        lat, lon = decode_geohash(gh)
        X[gh_to_idx[gh]] = [lat, lon, gh_counts.get(gh, 0.0)]
        
    X = (X - X.mean(axis=0)) / np.maximum(X.std(axis=0), 1e-6)
    X_t = torch.from_numpy(X).float().to(device)

    Y = np.zeros(N, dtype=np.float32)
    mask = np.zeros(N, dtype=np.bool_)
    
    # LEAKAGE FIX: Only use the strict training fold to calculate structural targets
    gh_means = train_target_df.groupby("geohash")[TARGET].mean().to_dict()
    g_mean = float(train_target_df[TARGET].mean())
    
    for gh in all_gh:
        if gh in gh_means:
            Y[gh_to_idx[gh]] = gh_means[gh]
            mask[gh_to_idx[gh]] = True
        else:
            Y[gh_to_idx[gh]] = g_mean
            
    Y_t = torch.from_numpy(Y).float().unsqueeze(1).to(device)
    mask_t = torch.from_numpy(mask).bool().to(device)

    model = GCN(in_features=3, hidden_dim=16).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=0.02)
    criterion = nn.MSELoss()

    model.train()
    for _ in range(150):
        opt.zero_grad()
        emb, pred = model(X_t, adj_t)
        loss = criterion(pred[mask_t], Y_t[mask_t])
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        final_emb, _ = model(X_t, adj_t)
        emb_np = final_emb.numpy()
        
    return {gh: emb_np[gh_to_idx[gh]] for gh in all_gh}


# --- 2. Oracle Feature Engineer ---
class OracleFeatureEngineer:
    def __init__(self, gnn_embeddings: dict[str, np.ndarray], smoothing: float = 10.0):
        self.gnn_embeddings = gnn_embeddings
        self.smoothing = smoothing
        self.road_enc = LabelEncoder()
        self.wea_enc = LabelEncoder()
        self.scaler = StandardScaler()
        self.global_demand = 0.5
        self.stats = {}
        self.num_cols = ["Temperature", "NumberofLanes", "lat", "lon", "lat_lon_product"]

    def fit(self, df: pd.DataFrame):
        work = self._basic(df)
        self.global_demand = float(work[TARGET].mean())
        
        self.road_enc.fit(work["RoadType"].astype(str))
        self.wea_enc.fit(work["Weather"].astype(str))
        
        # Calculate encodings
        for col in ["geohash", "geohash5", "geohash_hour", "geo5_hour"]:
            grp = work.groupby(col)[TARGET].agg(["mean", "count"])
            self.stats[col] = ((grp["count"] * grp["mean"] + self.smoothing * self.global_demand) / 
                               (grp["count"] + self.smoothing)).to_dict()
            
        feat = self.transform(df)
        self.scaler.fit(feat[self.num_cols].fillna(0))
        return self

    def transform(self, df: pd.DataFrame):
        work = self._basic(df)
        
        # Mappings
        for col in ["geohash", "geohash5", "geohash_hour", "geo5_hour"]:
            work[f"{col}_te"] = work[col].map(self.stats[col]).fillna(self.global_demand)
            
        work["is_gh_hour_unseen"] = work["geohash_hour"].map(lambda x: 0 if x in self.stats["geohash_hour"] else 1)
        work["RoadType"] = self.road_enc.transform(work["RoadType"].astype(str))
        work["Weather"] = self.wea_enc.transform(work["Weather"].astype(str))
        
        # GNN Embeddings
        gnn_feats = np.zeros((len(work), 16), dtype=np.float32)
        for i, gh in enumerate(work["geohash"]):
            gnn_feats[i] = self.gnn_embeddings.get(gh, np.zeros(16))
            
        for dim in range(16):
            work[f"gnn_emb_{dim}"] = gnn_feats[:, dim]
            
        cols = ["day", "hour", "minute_of_day", "hour_sin", "hour_cos", "is_rush_hour", 
                "is_night", "Temperature", "NumberofLanes", "lat", "lon", "lat_lon_product",
                "geohash_te", "geohash5_te", "geohash_hour_te", "geo5_hour_te", "is_gh_hour_unseen",
                "RoadType", "Weather"] + [f"gnn_emb_{d}" for d in range(16)]
        
        res = work[cols].copy().fillna(0)
        if hasattr(self.scaler, "mean_"):
            res[self.num_cols] = self.scaler.transform(res[self.num_cols])
        return res

    def _basic(self, df: pd.DataFrame):
        work = df.copy()
        work["hour"] = work["timestamp"].str.split(":").str[0].astype(int)
        work["minute"] = work["timestamp"].str.split(":").str[1].astype(int)
        coords = pd.DataFrame(work["geohash"].apply(decode_geohash).tolist(), index=work.index, columns=["lat", "lon"])
        work["lat"] = coords["lat"]
        work["lon"] = coords["lon"]
        work["lat_lon_product"] = work["lat"] * work["lon"]
        work["minute_of_day"] = work["hour"] * 60 + work["minute"]
        work["hour_sin"] = np.sin(2 * np.pi * work["hour"] / 24)
        work["hour_cos"] = np.cos(2 * np.pi * work["hour"] / 24)
        work["is_rush_hour"] = work["hour"].between(7, 9) | work["hour"].between(17, 19)
        work["is_night"] = (work["hour"] >= 22) | (work["hour"] <= 6)
        
        work["geohash5"] = work["geohash"].astype(str).str[:5]
        work["geo5_hour"] = work["geohash5"] + "__h" + work["hour"].astype(str)
        work["geohash_hour"] = work["geohash"].astype(str) + "__h" + work["hour"].astype(str)
        
        work["RoadType"] = work["RoadType"].fillna("Unknown")
        work["Weather"] = work["Weather"].fillna("Unknown")
        work["Temperature"] = work["Temperature"].fillna(25.0)
        return work


# --- 3. Zero-Inflated Model ---
class OracleTwoStageModel:
    def __init__(self):
        self.clf = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, verbose=-1, random_state=RANDOM_STATE)
        self.reg = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05, num_leaves=127, verbose=-1, random_state=RANDOM_STATE)

    def fit(self, X, y):
        print("  -> Fitting Classifier...")
        y_bin = (y > ZERO_THRESHOLD).astype(int)
        self.clf.fit(X, y_bin, categorical_feature=["RoadType", "Weather"])
        
        print("  -> Fitting Regressor...")
        active_mask = y > ZERO_THRESHOLD
        self.reg.fit(X[active_mask], y[active_mask], categorical_feature=["RoadType", "Weather"])

    def predict(self, X):
        prob = self.clf.predict_proba(X)[:, 1]
        vol = self.reg.predict(X)
        return np.clip(prob * vol, 0.0, 1.0)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", type=Path, default=Path("dataset/train.csv"))
    parser.add_argument("--test", type=Path, default=Path("dataset/test.csv"))
    parser.add_argument("--output", type=Path, default=Path("artifacts/gemini_v14_oracle.csv"))
    args = parser.parse_args()

    seed_everything(RANDOM_STATE)
    df = pd.read_csv(args.train).sort_values(["day", "timestamp"])
    test_df = pd.read_csv(args.test)
    
    print("\n--- Training V14 Oracle Pipeline (Honest Chronological Split) ---")
    
    # Custom chronological split to EXACTLY mimic the test set conditions.
    # Train set has Day 48 (all) and Day 49 (up to 02:45).
    # We will split Day 49 at hour 01:00.
    # Train fold: Day 48 (all) + Day 49 (< 01:00)
    # Valid fold: Day 49 (>= 01:00)
    
    hour_series = df["timestamp"].str.split(":").str[0].astype(int)
    
    train_mask = (df["day"] == 48) | ((df["day"] == 49) & (hour_series < 1))
    valid_mask = (df["day"] == 49) & (hour_series >= 1)
    
    tr_df = df[train_mask].copy()
    va_df = df[valid_mask].copy()
    
    # 1. GNN Embeddings (Strictly fitted on train fold only)
    gnn_emb_cv = get_gnn_embeddings(df, test_df, tr_df)

    fe = OracleFeatureEngineer(gnn_emb_cv).fit(tr_df)
    X_tr, X_val = fe.transform(tr_df), fe.transform(va_df)
    
    model = OracleTwoStageModel()
    model.fit(X_tr, tr_df[TARGET])
    preds = model.predict(X_val)
    
    score = r2_score(va_df[TARGET], preds)
    print(f"Honest Chronological CV R2: {score:.6f}")

    print("\nTraining Final Oracle Model on all data...")
    # For final model, GNN targets use all available training data
    gnn_emb_full = get_gnn_embeddings(df, test_df, df)
    
    fe = OracleFeatureEngineer(gnn_emb_full).fit(df)
    model = OracleTwoStageModel()
    model.fit(fe.transform(df), df[TARGET])
    
    test_preds = model.predict(fe.transform(test_df))
    pd.DataFrame({
        "Index": test_df["Index"], 
        "demand": test_preds
    }).to_csv(args.output, index=False)
    print(f"Saved to {args.output}")

if __name__ == "__main__":
    main()
